# imports

In [ ]:
import sys
sys.path.append(r'\\192.168.10.106\imdea\DataDriven_UT_AlbertoVicente\10_code\UTvsXCT-preprocessing')
from preprocess_tools import onlypores as onlypores_module, io, aligner, reslicer
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
import json
import time
import traceback

# volume list

List every XCT volume to process. Each volume is loaded, resliced and its walls located **once**, then `onlypores` is run on it once per parameter set below. Same format as `onlypores_batch.ipynb`.

In [ ]:
probetas_dir = Path(r'/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Fabricacion Nacho/05_Probetas_Nacho_2025/probetas')

A_values = ['01', '03', '04', '05', '06', '07', '08', '09', '10']
B_values = [1, 2, 3, 4, 5]

def volume_path_for(folder, stem='volume_eq_aligned'):
    """Use <stem>_rescaled.tif if the folder contains any file with 'rescaled' in its name, else <stem>.tif."""
    if any(f.is_file() and 'rescaled' in f.name for f in folder.iterdir()):
        return folder / f'{stem}_rescaled.tif'
    return folder / f'{stem}.tif'

volume_paths = [
    volume_path_for(probetas_dir / f'Na_{A}_{B}')
    for A in A_values
    for B in B_values
]

n_rescaled = sum('rescaled' in p.name for p in volume_paths)
missing = [p for p in volume_paths if not p.exists()]
print(f"{len(volume_paths)} volumes ({n_rescaled} rescaled), {len(missing)} missing")
for p in missing:
    print(f"  missing: {p}")

# parameter sets

Each entry is one set of `onlypores` parameters. `label` is optional (defaults to `r<radius>_k<k>_min<min_size>`) and must be unique. **The first set is the reference**: all deltas in the comparison are computed against it. At most 8 sets (one fixed plot colour per set).

In [ ]:
param_sets = [
    {"label": "baseline", "sauvola_radius": 15, "sauvola_k": 0.2,   "min_size_filtering": 8},
    {"label": "r30_k0.125", "sauvola_radius": 30, "sauvola_k": 0.125, "min_size_filtering": 8}
]

# output

Directory where the comparison tables (`.csv`), plots (`.png`) and `config.json` are written. **Required.** No volumes / masks are saved.

In [ ]:
output_dir = "/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Fabricacion Nacho/05_Probetas_Nacho_2025/probetas/onlypores comparison"   # <-- set this, e.g. r'/path/to/onlypores_comparison'

if not str(output_dir).strip():
    raise ValueError("output_dir is empty: set it in this cell before running the notebook.")
output_dir = Path(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

# normalise parameter sets: fill default labels, check uniqueness and the colour limit
PARAM_KEYS = ("sauvola_radius", "sauvola_k", "min_size_filtering")
param_sets = [dict(p) for p in param_sets]
for p in param_sets:
    missing = [k for k in PARAM_KEYS if k not in p]
    if missing:
        raise ValueError(f"Parameter set {p} is missing {missing}")
    p.setdefault("label", f"r{p['sauvola_radius']}_k{p['sauvola_k']}_min{p['min_size_filtering']}")
labels = [p["label"] for p in param_sets]
if len(set(labels)) != len(labels):
    raise ValueError(f"Parameter set labels must be unique: {labels}")
if not (1 <= len(labels) <= 8):
    raise ValueError(f"Between 1 and 8 parameter sets supported, got {len(labels)}")
reference_label = labels[0]

# short unique name per volume (parent folder; parent/stem if two volumes share a folder name)
volume_names = {}
for vp in volume_paths:
    name = vp.parent.name
    volume_names[vp] = name
if len(set(volume_names.values())) != len(volume_names):
    volume_names = {vp: f"{vp.parent.name}/{vp.stem}" for vp in volume_paths}

(output_dir / "config.json").write_text(json.dumps({
    "generated": datetime.now().isoformat(timespec="seconds"),
    "reference_label": reference_label,
    "volume_paths": [str(v) for v in volume_paths],
    "param_sets": param_sets,
}, indent=2), encoding="utf-8")
print(f"Output directory: {output_dir}")
print(f"{len(volume_paths)} volume(s) x {len(param_sets)} parameter set(s) = {len(volume_paths) * len(param_sets)} onlypores runs (reference: '{reference_label}')")

# processing

For each volume, `onlypores` is run with every parameter set and only the VVF statistics are kept (pore / sample-mask / binary arrays are discarded right away). A failure on one (volume, set) is logged and skipped. `results_long.csv` is rewritten after every run, so partial results survive a crash.

In [ ]:
def prepare_volume(volume_path):
    """Load, reslice and locate the walls of a volume (done once per volume)."""
    volume = io.load_tif(volume_path)
    print(f"Loaded volume shape: {volume.shape}")
    resliced_volume = reslicer.rotate_90(volume, False)
    resliced_volume = reslicer.reslice(resliced_volume, 'Right')
    _, frontwall, backwall = aligner.crop_walls(resliced_volume)
    print(f"Front wall: {frontwall}, back wall: {backwall}")
    del resliced_volume
    return volume, frontwall, backwall


def run_param_set(volume, frontwall, backwall, p):
    """Run onlypores with one parameter set and return the VVF statistics (no arrays kept)."""
    t0 = time.perf_counter()
    pores, sample_mask, binary = onlypores_module.onlypores(
        volume, frontwall, backwall,
        sauvola_radius=p["sauvola_radius"], sauvola_k=p["sauvola_k"],
        min_size_filtering=p["min_size_filtering"]
    )
    runtime_s = time.perf_counter() - t0
    pore_voxels = int(np.sum(pores))
    material_voxels = int(np.sum(sample_mask))
    del pores, sample_mask, binary
    vvf = pore_voxels / material_voxels if material_voxels > 0 else float('nan')
    return {"pore_voxels": pore_voxels, "material_voxels": material_voxels, "vvf": vvf, "runtime_s": runtime_s}


rows = []
errors = []

for volume_path in volume_paths:
    print(f"\n{'='*80}\nVolume: {volume_path}\n{'='*80}")
    try:
        volume, frontwall, backwall = prepare_volume(volume_path)
    except Exception as e:
        print(f"ERROR preparing {volume_path}: {e}")
        traceback.print_exc()
        errors.append((volume_path, "all sets", e))
        continue

    for p in param_sets:
        print(f"\n--- {volume_names[volume_path]} | {p['label']}: radius={p['sauvola_radius']}, k={p['sauvola_k']}, min_size={p['min_size_filtering']}")
        try:
            stats = run_param_set(volume, frontwall, backwall, p)
        except Exception as e:
            print(f"ERROR on {volume_path} with '{p['label']}': {e}")
            traceback.print_exc()
            errors.append((volume_path, p["label"], e))
            continue
        rows.append({
            "volume": volume_names[volume_path],
            "volume_path": str(volume_path),
            "label": p["label"],
            **{k: p[k] for k in PARAM_KEYS},
            "frontwall": int(frontwall),
            "backwall": int(backwall),
            **stats,
        })
        print(f"VVF = {stats['vvf']*100:.4f}%  ({stats['runtime_s']:.0f} s)")
        pd.DataFrame(rows).to_csv(output_dir / "results_long.csv", index=False)

    del volume

print(f"\n{'='*80}\nDone: {len(rows)} runs succeeded, {len(errors)} failed\n{'='*80}")
for volume_path, label, e in errors:
    print(f"  FAIL  {volume_path}  [{label}]  ({e})")

# comparison tables

- `vvf_pivot.csv`: VVF (%) per volume (rows) and parameter set (columns)
- `delta_vs_reference.csv`: per volume and set, VVF difference vs the reference set (percentage points and relative %)
- `summary_by_set.csv`: VVF statistics across volumes for each set

In [ ]:
if not rows:
    raise RuntimeError("No successful runs, nothing to compare.")

df = pd.DataFrame(rows)
df["vvf_pct"] = df["vvf"] * 100

volume_order = list(dict.fromkeys(df["volume"]))
label_order = [l for l in labels if l in set(df["label"])]

pivot = df.pivot(index="volume", columns="label", values="vvf_pct").reindex(index=volume_order, columns=label_order)

if reference_label in pivot.columns:
    ref = pivot[reference_label]
else:
    print(f"WARNING: reference set '{reference_label}' has no successful runs, deltas will be NaN")
    ref = pd.Series(np.nan, index=pivot.index)
delta_pp = pivot.sub(ref, axis=0)
delta_rel = delta_pp.div(ref, axis=0) * 100

delta_long = (
    pd.DataFrame({"vvf_pct": pivot.stack(), "delta_pp": delta_pp.stack(), "delta_rel_pct": delta_rel.stack()})
    .rename_axis(["volume", "label"]).reset_index()
)

summary = df.groupby("label").agg(
    n_volumes=("vvf_pct", "count"),
    mean_vvf_pct=("vvf_pct", "mean"),
    std_vvf_pct=("vvf_pct", "std"),
    min_vvf_pct=("vvf_pct", "min"),
    median_vvf_pct=("vvf_pct", "median"),
    max_vvf_pct=("vvf_pct", "max"),
    mean_runtime_s=("runtime_s", "mean"),
).reindex(label_order)
summary["mean_abs_delta_pp_vs_ref"] = delta_pp.abs().mean().reindex(label_order)

pivot.to_csv(output_dir / "vvf_pivot.csv")
delta_long.to_csv(output_dir / "delta_vs_reference.csv", index=False)
summary.to_csv(output_dir / "summary_by_set.csv")
df.to_csv(output_dir / "results_long.csv", index=False)

print("VVF (%) per volume and parameter set:")
display(pivot.round(4))
print(f"Delta vs reference '{reference_label}' (percentage points):")
display(delta_pp.round(4))
print("Summary by parameter set:")
display(summary.round(4))

# comparison plots

One fixed colour per parameter set (same colour in every plot). Saved as `.png` in the output directory.

In [ ]:
# categorical palette, fixed order (one slot per parameter set, never cycled)
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
color_of = {l: PALETTE[i] for i, l in enumerate(labels)}
SURFACE, INK, INK_2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": INK_2, "xtick.color": INK_2, "ytick.color": INK_2,
    "axes.edgecolor": GRID, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.titleweight": "bold", "axes.titlesize": 12, "font.size": 10,
})


def finish(fig, name):
    fig.tight_layout()
    fig.savefig(output_dir / name, dpi=200)
    plt.show()
    print(f"Saved {output_dir / name}")


def grouped_bars(ax, table, series, ylabel):
    """Grouped bars: one group per volume, one bar per series (column of `table`)."""
    n = len(series)
    width = 0.8 / n
    x = np.arange(len(table.index))
    for i, s in enumerate(series):
        ax.bar(x + (i - (n - 1) / 2) * width, table[s].values, width, label=s,
               color=color_of[s], edgecolor=SURFACE, linewidth=1.5)
    ax.set_xticks(x)
    ax.set_xticklabels(table.index, rotation=45 if len(x) > 4 else 0, ha="right" if len(x) > 4 else "center")
    ax.set_ylabel(ylabel)
    ax.grid(axis="x", visible=False)
    ax.legend(title="parameter set", frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1))


fig_w = max(7, 0.9 * len(volume_order) * len(label_order) * 0.5 + 3)

# 1. VVF per volume, grouped by parameter set
fig, ax = plt.subplots(figsize=(fig_w, 4.5))
grouped_bars(ax, pivot, label_order, "VVF (%)")
ax.set_title("VVF per volume and parameter set", loc="left")
finish(fig, "vvf_per_volume.png")

# 2. VVF distribution across volumes for each parameter set
fig, ax = plt.subplots(figsize=(max(6, 1.3 * len(label_order) + 2), 4.5))
rng = np.random.default_rng(0)
for i, l in enumerate(label_order):
    vals = pivot[l].dropna().values
    if len(vals) >= 2:
        bp = ax.boxplot(vals, positions=[i], widths=0.5, patch_artist=True, showfliers=False,
                        medianprops=dict(color=INK, linewidth=1.5), whiskerprops=dict(color=INK_2), capprops=dict(color=INK_2),
                        boxprops=dict(edgecolor=INK_2))
        bp["boxes"][0].set_facecolor(color_of[l] + "55")
    ax.scatter(i + rng.uniform(-0.1, 0.1, len(vals)), vals, s=45, color=color_of[l], edgecolor=SURFACE, linewidth=1.5, zorder=3)
ax.set_xticks(range(len(label_order)))
ax.set_xticklabels(label_order, rotation=30 if len(label_order) > 3 else 0, ha="right" if len(label_order) > 3 else "center")
ax.set_ylabel("VVF (%)")
ax.grid(axis="x", visible=False)
ax.set_title(f"VVF distribution across {len(volume_order)} volume(s)", loc="left")
finish(fig, "vvf_distribution_by_set.png")

# 3. difference vs the reference set, per volume
other_labels = [l for l in label_order if l != reference_label]
if other_labels:
    fig, ax = plt.subplots(figsize=(fig_w, 4.5))
    grouped_bars(ax, delta_pp, other_labels, f"VVF difference vs '{reference_label}' (percentage points)")
    ax.axhline(0, color=INK_2, linewidth=1)
    ax.set_title("Effect of parameters on VVF (vs reference set)", loc="left")
    finish(fig, "delta_vs_reference.png")
else:
    print("Only the reference set was run: skipping the delta plot.")

# 4. heatmap volumes x parameter sets
fig, ax = plt.subplots(figsize=(max(5, 1.4 * len(label_order) + 2), max(3, 0.5 * len(volume_order) + 1.5)))
data = pivot.values.astype(float)
im = ax.imshow(data, cmap="Blues", aspect="auto")
ax.set_xticks(range(len(label_order)))
ax.set_xticklabels(label_order, rotation=30 if len(label_order) > 3 else 0, ha="right" if len(label_order) > 3 else "center")
ax.set_yticks(range(len(volume_order)))
ax.set_yticklabels(volume_order)
ax.grid(False)
vmin, vmax = np.nanmin(data), np.nanmax(data)
for r in range(data.shape[0]):
    for c in range(data.shape[1]):
        if np.isnan(data[r, c]):
            continue
        dark_cell = vmax > vmin and (data[r, c] - vmin) / (vmax - vmin) > 0.6
        ax.text(c, r, f"{data[r, c]:.3f}", ha="center", va="center", color="white" if dark_cell else INK, fontsize=9)
fig.colorbar(im, ax=ax, label="VVF (%)")
ax.set_title("VVF (%) heatmap", loc="left")
finish(fig, "vvf_heatmap.png")

# summary

In [ ]:
print(f"{len(rows)} runs OK, {len(errors)} failed. No volumes were saved.")
print(f"Results in: {output_dir}")
for f in sorted(output_dir.iterdir()):
    print(f"  {f.name}")